# NLP

NLP, ou Processamento de Linguagem Natural, e uma area da ciencia de dados que estuda como transformar textos em informacao estruturada. Com essas tecnicas, conseguimos analisar noticias, identificar temas, contar palavras, comparar documentos e preparar dados textuais para modelos de aprendizado de maquina.

In [10]:
import pandas as pd

df = pd.read_csv("../Webscraping/data/noticias_palmeiras (1).csv", sep = ";")
df.head()

df['texto'] = df['texto'].astype(str)


## Trabalhando apenas com o texto

Por enquanto, vamos trabalhar somente com a coluna `texto`, que contem o corpo completo de cada noticia. As outras colunas, como `titulo`, `descricao`, `tags` e `data`, continuam no `DataFrame`, mas vamos deixar elas de lado neste primeiro momento para entender melhor o conteudo textual.

## Passos da analise

Vamos preparar os textos aos poucos:

1. Limpar os textos.
2. Remover palavras muito comuns.
3. Criar uma representacao Bag of Words.
4. Contar palavras frequentes.
5. Transformar textos em numeros para analises posteriores.

## 1. Limpeza basica

Na celula abaixo:

- `wordpunct_tokenize` separa o texto em palavras e pontuacao.
- `texto.lower()` coloca tudo em minusculas.
- `unidecode(texto)` troca letras acentuadas por letras sem acento.
- `token.isalnum()` mantem apenas letras e numeros.
- `" ".join(tokens)` junta os tokens em uma frase limpa.
- `.apply(limpar_texto)` aplica a funcao em todas as noticias.

Exemplo: `"Olá, Senado!"` vira `"ola senado"`.

In [11]:
from nltk.tokenize import wordpunct_tokenize
from unidecode import unidecode


def limpar_texto(texto):
    texto = texto.lower()
    texto = unidecode(texto)
    tokens = wordpunct_tokenize(texto)
    tokens = [token for token in tokens if token.isalnum()]
    return " ".join(tokens)


df["texto_limpo"] = df["texto"].apply(limpar_texto)

df[["texto", "texto_limpo"]].head()

,texto,texto_limpo
0,O Palmeiras terá uma nova rodada de desafios n...,o palmeiras tera uma nova rodada de desafios n...
1,O Nubank está explorando a possibilidade de or...,o nubank esta explorando a possibilidade de or...
2,A Sociedade Esportiva Palmeiras manifestou sua...,a sociedade esportiva palmeiras manifestou sua...
3,"Na manhã deste sábado (11), a Academia de Fute...",na manha deste sabado 11 a academia de futebol...
4,"A co-fundadora do Nubank, Cristina Junqueira, ...",a co fundadora do nubank cristina junqueira re...


## 2. Removendo stopwords

Stopwords sao palavras muito comuns, como `a`, `o`, `de`, `para` e `que`. Elas aparecem muito, mas geralmente ajudam pouco a entender o tema de um texto.

Na celula abaixo:

- `stopwords.words("portuguese")` carrega stopwords em portugues.
- `texto.split()` separa o texto limpo em palavras.
- `token not in stopwords_pt` remove as palavras muito comuns.
- `.str.join(" ")` junta os tokens restantes em um texto sem stopwords.
- `.apply(remover_stopwords)` aplica a funcao em todas as noticias.

Exemplo: `"o senador falou com a imprensa"` vira `["senador", "falou", "imprensa"]`.

In [12]:
import nltk

from nltk.corpus import stopwords


nltk.download("stopwords", quiet=True)

stopwords_pt = stopwords.words("portuguese")
stopwords_pt = [unidecode(palavra) for palavra in stopwords_pt]
stopwords_pt = set(stopwords_pt)


def remover_stopwords(texto):
    tokens = texto.split()
    tokens = [token for token in tokens if token not in stopwords_pt]
    return tokens


df["tokens_sem_stopwords"] = df["texto_limpo"].apply(remover_stopwords)
df["texto_sem_stopwords"] = df["tokens_sem_stopwords"].str.join(" ")

df[["texto_limpo", "tokens_sem_stopwords", "texto_sem_stopwords"]].head()

,texto_limpo,tokens_sem_stopwords,texto_sem_stopwords
0,o palmeiras tera uma nova rodada de desafios n...,"[palmeiras, nova, rodada, desafios, campeonato...",palmeiras nova rodada desafios campeonato bras...
1,o nubank esta explorando a possibilidade de or...,"[nubank, explorando, possibilidade, organizar,...",nubank explorando possibilidade organizar amis...
2,a sociedade esportiva palmeiras manifestou sua...,"[sociedade, esportiva, palmeiras, manifestou, ...",sociedade esportiva palmeiras manifestou insat...
3,na manha deste sabado 11 a academia de futebol...,"[manha, deste, sabado, 11, academia, futebol, ...",manha deste sabado 11 academia futebol palco u...
4,a co fundadora do nubank cristina junqueira re...,"[co, fundadora, nubank, cristina, junqueira, r...",co fundadora nubank cristina junqueira revelou...


## 3. Bag of Words

No Bag of Words, cada linha representa uma noticia e cada coluna representa uma palavra. O valor indica quantas vezes aquela palavra apareceu na noticia.

Exemplo: `"senador falou senador"` teria `senador = 2` e `falou = 1`.

In [14]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
matriz_bow = vectorizer.fit_transform(df["texto_sem_stopwords"])

df_bow = pd.DataFrame(
    matriz_bow.toarray(),
    columns=vectorizer.get_feature_names_out()
)

df_bow.head()

,00,000,00central,00para,00r,00superior,01,012,02,03,...,zerar,zero,zhu,zie,zona,zonas,zortea,zorzi,zubeldia,zvlr9u0adq
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Removendo colunas com numeros

Agora vamos remover qualquer coluna cujo nome tenha pelo menos um numero.

In [15]:
colunas_com_numeros = [col for col in df_bow.columns if any(char.isdigit() for char in col)]

df_bow = df_bow.drop(columns=colunas_com_numeros)

print(f"{len(colunas_com_numeros)} colunas removidas")
df_bow.head()

505 colunas removidas


,aa,abafa,abafado,abafadora,abaixo,abala,abalada,abalado,abalar,abalo,...,zerados,zerar,zero,zhu,zie,zona,zonas,zortea,zorzi,zubeldia
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Criando o DataFrame final

Agora vamos juntar os metadados das noticias com as colunas do Bag of Words. Para evitar conflito de nomes, as colunas do Bag of Words recebem o prefixo `bow_`.

In [17]:
metadados = df[["data", "titulo"]].reset_index(drop=True)
bow_com_prefixo = df_bow.add_prefix("bow_").reset_index(drop=True)

df_final = pd.concat([metadados, bow_com_prefixo], axis=1)

df_final.head()

,data,titulo,bow_aa,bow_abafa,bow_abafado,bow_abafadora,bow_abaixo,bow_abala,bow_abalada,bow_abalado,...,bow_zerados,bow_zerar,bow_zero,bow_zhu,bow_zie,bow_zona,bow_zonas,bow_zortea,bow_zorzi,bow_zubeldia
0,11/4/2026 16:47,STJD indefere pedido de suspensão e Abel Ferre...,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,11/4/2026 16:47,Nubank Anuncia Amistoso Imperdível: Palmeiras ...,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,11/4/2026 16:47,STJD Recusa Suspensão de Abel Ferreira; Palmei...,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,11/4/2026 14:27,Palmeiras encerra treinos para o Dérbi com Vit...,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,11/4/2026 14:07,Messi Pode Anunciar Amistoso Histórico em São ...,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Exemplo de analise

Agora podemos fazer uma analise simples das palavras do Bag of Words: quais aparecem mais, quais aparecem menos e quantas palavras diferentes temos no vocabulario.

In [18]:
colunas_bow = [coluna for coluna in df_final.columns if coluna.startswith("bow_")]

frequencia_palavras = df_final[colunas_bow].sum().sort_values(ascending=False)

print(f"Total de palavras diferentes: {len(frequencia_palavras)}")

frequencia_palavras.head(10)

Total de palavras diferentes: 16336


bow_palmeiras     8159
bow_equipe        3782
bow_jogo          2869
bow_campeonato    2742
bow_abel          2598
bow_clube         2353
bow_elenco        2283
bow_ferreira      2203
bow_apos          1964
bow_desempenho    1947
dtype: int64

In [19]:
frequencia_palavras.tail(10)

bow_badaladas         1
bow_bafometro         1
bow_jurisprudencia    1
bow_bagunca           1
bow_juridicamente     1
bow_baguncar          1
bow_junxpal           1
bow_bahxpal           1
bow_baianas           1
bow_aa                1
dtype: int64

In [20]:
documentos_por_palavra = (df_final[colunas_bow] > 0).sum().sort_values(ascending=False)
documentos_por_palavra.index = documentos_por_palavra.index.str.replace("bow_", "", regex=False)

documentos_por_palavra.head(10)

palmeiras     1960
elenco        1449
equipe        1446
jogo          1429
campeonato    1373
gestao        1334
desempenho    1256
apos          1249
abel          1201
ferreira      1199
dtype: int64

In [21]:
df_final["palavras_unicas"] = (df_final[colunas_bow] > 0).sum(axis=1)

df_final[["titulo", "palavras_unicas"]].sort_values("palavras_unicas", ascending=True).head(10)

,titulo,palavras_unicas
275,Melhores momentos! Veja os gols da vitória,1
1533,Palmeiras x Guarani: melhores momentos,1
1407,Palmeiras x Capivariano: melhores momentos,1
967,Novorizontino x Palmeiras: melhores momentos,1
729,Melhores momentos!! Confira o golaçõ e os mome...,1
190,OLHO NA TABELA! Palmeiras segue na liderança a...,1
640,ESCALAÇÃO! Palmeiras pronto e escalado para en...,1
1203,Palmeiras x São Paulo: melhores momentos,1
1723,OLHO NA TABELA! Palmeiras se aproxima da lider...,1
1724,Corinthians x Palmeiras: melhores momentos,1
